Puedes pedir las versiones impresa y ebook de *Think Python 3e* en
[Bookshop.org](https://bookshop.org/a/98697/9781098155438) y
[Amazon](https://www.amazon.com/_/dp/1098155432?smid=ATVPDKIKX0DER&_encoding=UTF8&tag=oreilly20-20&_encoding=UTF8&tag=greenteapre01-20&linkCode=ur2&linkId=e2a529f94920295d27ec8a06e757dc7c&camp=1789&creative=9325).

In [1]:
from os.path import basename, exists

def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve

        local, _ = urlretrieve(url, filename)
        print("Downloaded " + str(local))
    return filename

download('https://github.com/AllenDowney/ThinkPython/raw/v3/thinkpython.py');
download('https://github.com/AllenDowney/ThinkPython/raw/v3/diagram.py');

import thinkpython

Downloaded thinkpython.py
Downloaded diagram.py


# Análisis y generación de texto

A estas alturas hemos cubierto las estructuras de datos principales de Python -- listas, diccionarios y tuplas -- y algunos algoritmos que las usan.
En este capítulo, las usaremos para explorar el análisis de texto y la generación de Markov:

* El análisis de texto es una forma de describir las relaciones estadísticas entre las palabras de un documento, como la probabilidad de que una palabra vaya seguida de otra, y

* La generación de Markov es una forma de generar texto nuevo con palabras y frases similares a las del texto original.

Estos algoritmos son similares a partes de un Large Language Model (LLM), que es el componente clave de un chatbot.

Empezaremos contando el número de veces que aparece cada palabra en un libro.
Luego veremos pares de palabras y haremos una lista de las palabras que pueden seguir a cada palabra.
Haremos una versión simple de un generador de Markov y, como ejercicio, tendrás la oportunidad de hacer una versión más general.

## Palabras únicas

Como primer paso hacia el análisis de texto, leamos un libro -- *The Strange Case Of Dr. Jekyll And Mr. Hyde* de Robert Louis Stevenson -- y contemos el número de palabras únicas.
Las instrucciones para descargar el libro están en el notebook de este capítulo.

La siguiente celda descarga el libro desde Project Gutenberg.

In [2]:
download('https://www.gutenberg.org/cache/epub/43/pg43.txt');

Downloaded pg43.txt


La versión disponible en Project Gutenberg incluye información sobre el libro al principio e información de licencia al final.
Usaremos `clean_file` del Capítulo 8 para eliminar este material y escribir un archivo "limpio" que contiene solo el texto del libro.

In [3]:
def is_special_line(line):
    return line.strip().startswith('*** ')

In [4]:
def clean_file(input_file, output_file):
    reader = open(input_file, encoding='utf-8')
    writer = open(output_file, 'w')

    for line in reader:
        if is_special_line(line):
            break

    for line in reader:
        if is_special_line(line):
            break
        writer.write(line)

    reader.close()
    writer.close()

In [5]:
filename = 'dr_jekyll.txt'

In [6]:
clean_file('pg43.txt', filename)

Usaremos un bucle `for` para leer líneas del archivo y `split` para dividir las líneas en palabras.
Luego, para llevar la cuenta de las palabras únicas, guardaremos cada palabra como una clave en un diccionario.

In [7]:
unique_words = {}
for line in open(filename):
    seq = line.split()
    for word in seq:
        unique_words[word] = 1

len(unique_words)

6039

La longitud del diccionario es el número de palabras únicas -- unas `6000` con esta forma de contar.
Pero si las inspeccionamos, veremos que algunas no son palabras válidas.

Por ejemplo, veamos las palabras más largas en `unique_words`.
Podemos usar `sorted` para ordenar las palabras, pasando la función `len` como keyword argumento para que las palabras se ordenen por longitud.

In [8]:
sorted(unique_words, key=len)[-5:]

['chocolate-coloured',
 'superiors—behold!”',
 'coolness—frightened',
 'gentleman—something',
 'pocket-handkerchief.']

El índice de porción, `[-5:]`, selecciona los últimos `5` elementos de la lista ordenada, que son las palabras más largas.

La lista incluye algunas palabras legítimamente largas, como "circumscription", y algunas palabras con guion, como "chocolate-coloured".
Pero algunas de las "palabras" más largas son en realidad dos palabras separadas por una raya.
Y otras palabras incluyen puntuación como puntos, signos de exclamación y comillas.

Así que, antes de seguir, ocupémonos de las rayas y otros signos de puntuación.

## Puntuación

Para identificar las palabras del texto, tenemos que resolver dos cuestiones:

* Cuando aparece una raya en una línea, deberíamos reemplazarla por un espacio -- así, cuando usemos `split`, las palabras quedarán separadas.

* Después de separar las palabras, podemos usar `strip` para eliminar la puntuación.

Para manejar la primera cuestión, podemos usar la siguiente función, que toma un string, reemplaza las rayas por espacios, divide el string y devuelve la lista resultante.

In [9]:
def split_line(line):
    return line.replace('—', ' ').split()

Observa que `split_line` solo reemplaza rayas, no guiones.
Aquí tienes un ejemplo.

In [10]:
split_line('coolness—frightened')

['coolness', 'frightened']

Ahora, para eliminar la puntuación del principio y del final de cada palabra, podemos usar `strip`, pero necesitamos una lista de caracteres que se consideran puntuación.

Los caracteres en los strings de Python están en Unicode, que es un estándar internacional usado para representar letras de casi todos los alfabetos, números, símbolos, signos de puntuación y más.
El módulo `unicodedata` proporciona una función `category` que podemos usar para saber qué caracteres son puntuación.
Dada una letra, devuelve un string con información sobre la categoría a la que pertenece.

In [11]:
import unicodedata

unicodedata.category('A')

'Lu'

El string de categoría de `'A'` es `'Lu'` -- la `'L'` significa que es una letra y la `'u'` significa que es mayúscula.

El string de categoría de `'.'` es `'Po'` -- la `'P'` significa que es puntuación y la `'o'` significa que su subcategoría es "other".

In [12]:
unicodedata.category('.')

'Po'

Podemos encontrar los signos de puntuación del libro comprobando los caracteres con categorías que empiezan por `'P'`.
El siguiente bucle guarda los signos de puntuación únicos en un diccionario.

In [13]:
punc_marks = {}
for line in open(filename):
    for char in line:
        category = unicodedata.category(char)
        if category.startswith('P'):
            punc_marks[char] = 1

Para crear una lista de signos de puntuación, podemos unir las claves del diccionario en un string.

In [14]:
punctuation = ''.join(punc_marks)
print(punctuation)

.’;,-“”:?—‘!()_


Ahora que sabemos qué caracteres del libro son puntuación, podemos escribir una función que toma una palabra, elimina la puntuación del principio y del final, y la convierte a minúsculas.

In [15]:
def clean_word(word):
    return word.strip(punctuation).lower()

Aquí tienes un ejemplo.

In [16]:
clean_word('“Behold!”')

'behold'

Como `strip` elimina caracteres del principio y del final, deja intactas las palabras con guion.

In [17]:
clean_word('pocket-handkerchief')

'pocket-handkerchief'

Ahora aquí tienes un bucle que usa `split_line` y `clean_word` para identificar las palabras únicas del libro.

In [18]:
unique_words2 = {}
for line in open(filename):
    for word in split_line(line):
        word = clean_word(word)
        unique_words2[word] = 1

len(unique_words2)

4003

Con esta definición más estricta de lo que es una palabra, hay unas 4000 palabras únicas.
Y podemos confirmar que la lista de palabras más largas se ha limpiado.

In [19]:
sorted(unique_words2, key=len)[-5:]

['circumscription',
 'unimpressionable',
 'fellow-creatures',
 'chocolate-coloured',
 'pocket-handkerchief']

Ahora veamos cuántas veces se usa cada palabra.

## Frecuencias de palabras

El siguiente bucle calcula la frecuencia de cada palabra única.

In [20]:
word_counter = {}
for line in open(filename):
    for word in split_line(line):
        word = clean_word(word)
        if word not in word_counter:
            word_counter[word] = 1
        else:
            word_counter[word] += 1

La primera vez que vemos una palabra, inicializamos su frecuencia a `1`. Si volvemos a ver la misma palabra más adelante, incrementamos su frecuencia.

Para ver qué palabras aparecen con más frecuencia, podemos usar `items` para obtener los pares clave-valor de `word_counter`, y ordenarlos por el segundo elemento del par, que es la frecuencia.
Primero definiremos una función que selecciona el segundo elemento.

In [21]:
def second_element(t):
    return t[1]

Ahora podemos usar `sorted` con dos keyword argumentos:

* `key=second_element` significa que los elementos se ordenarán según las frecuencias de las palabras.

* `reverse=True` significa que los elementos se ordenarán en orden inverso, con las palabras más frecuentes primero.

In [22]:
items = sorted(word_counter.items(), key=second_element, reverse=True)

Aquí están las cinco palabras más frecuentes.

In [23]:
for word, freq in items[:5]:
    print(freq, word, sep='\t')

1614	the
972	and
941	of
640	to
640	i


En la siguiente sección, encapsularemos este bucle en una función.
Y la usaremos para demostrar una nueva característica -- parámetros opcionales.

## Parámetros opcionales

Hemos usado funciones integradas que toman parámetros opcionales.
Por ejemplo, `round` toma un parámetro opcional llamado `ndigits` que indica cuántos decimales conservar.

In [24]:
round(3.141592653589793, ndigits=3)

3.142

Pero no son solo las funciones integradas -- también podemos escribir funciones con parámetros opcionales.
Por ejemplo, la siguiente función toma dos parámetros, `word_counter` y `num`.

In [25]:
def print_most_common(word_counter, num=5):
    items = sorted(word_counter.items(), key=second_element, reverse=True)

    for word, freq in items[:num]:
        print(freq, word, sep='\t')

El segundo parámetro parece una sentencia de asignación, pero no lo es -- es un parámetro opcional.

Si llamas a esta función con un argumento, `num` recibe el **valor por defecto**, que es `5`.

In [26]:
print_most_common(word_counter)

1614	the
972	and
941	of
640	to
640	i


Si llamas a esta función con dos argumentos, el segundo argumento se asigna a `num` en lugar del valor por defecto.

In [27]:
print_most_common(word_counter, 3)

1614	the
972	and
941	of


En ese caso, diríamos que el argumento opcional **override** el valor por defecto.

Si una función tiene parámetros requeridos y opcionales, todos los parámetros requeridos tienen que ir primero, seguidos por los opcionales.

In [28]:
%%expect SyntaxError

def bad_function(n=5, word_counter):
    return None

SyntaxError: parameter without a default follows parameter with a default (3116647453.py, line 1)

## Resta de diccionarios

Supongamos que queremos revisar la ortografía de un libro -- es decir, encontrar una lista de palabras que podrían estar mal escritas.
Una forma de hacerlo es encontrar palabras del libro que no aparecen en una lista de palabras válidas.
En capítulos anteriores, hemos usado una lista de palabras que se consideran válidas en juegos de palabras como Scrabble.
Ahora usaremos esta lista para revisar la ortografía de Robert Louis Stevenson.

Podemos pensar en este problema como una resta de conjuntos -- es decir, queremos encontrar todas las palabras de un conjunto (las palabras del libro) que no están en el otro (las palabras de la lista).

La siguiente celda descarga la lista de palabras.

In [29]:
download('https://raw.githubusercontent.com/AllenDowney/ThinkPython/v3/words.txt');

Downloaded words.txt


Como hemos hecho antes, podemos leer el contenido de `words.txt` y dividirlo en una lista de strings.

In [30]:
word_list = open('words.txt').read().split()

Luego guardaremos las palabras como claves en un diccionario para poder usar el operador `in` y comprobar rápidamente si una palabra es válida.

In [31]:
valid_words = {}
for word in word_list:
    valid_words[word] = 1

Ahora, para identificar palabras que aparecen en el libro pero no en la lista de palabras, usaremos `subtract`, que toma dos diccionarios como parámetros y devuelve un nuevo diccionario que contiene todas las claves de uno que no están en el otro.

In [32]:
def subtract(d1, d2):
    res = {}
    for key in d1:
        if key not in d2:
            res[key] = d1[key]
    return res

Así es como lo usamos.

In [33]:
diff = subtract(word_counter, valid_words)

Para obtener una muestra de palabras que podrían estar mal escritas, podemos imprimir las palabras más comunes en `diff`.

In [34]:
print_most_common(diff)

640	i
628	a
128	utterson
124	mr
98	hyde


Las "palabras mal escritas" más comunes son en su mayoría nombres y algunas palabras de una sola letra (Mr. Utterson es el amigo y abogado del Dr. Jekyll).

Si seleccionamos palabras que solo aparecen una vez, es más probable que sean errores ortográficos reales.
Podemos hacerlo haciendo un bucle por los elementos y creando una lista de palabras con frecuencia `1`.

In [35]:
singletons = []
for word, freq in diff.items():
    if freq == 1:
        singletons.append(word)

Aquí están los últimos elementos de la lista.

In [36]:
singletons[-5:]

['gesticulated', 'abjection', 'circumscription', 'reindue', 'fearstruck']

La mayoría son palabras válidas que no están en la lista de palabras.
Pero `'reindue'` parece ser un error ortográfico de `'reinduce'`, así que al menos encontramos un error legítimo.

## Números aleatorios

Como paso hacia la generación de texto de Markov, a continuación elegiremos una secuencia aleatoria de palabras de `word_counter`.
Pero primero hablemos de aleatoriedad.

Dados los mismos inputs, la mayoría de los programas de computadora son **determinista**, lo que significa que generan los mismos outputs cada vez.
El determinismo suele ser algo bueno, ya que esperamos que el mismo cálculo produzca el mismo resultado.
Para algunas aplicaciones, sin embargo, queremos que la computadora sea impredecible.
Los juegos son un ejemplo, pero hay más.

Hacer que un programa sea verdaderamente no determinista resulta ser difícil, pero hay formas de simularlo.
Una es usar algoritmos que generan números **pseudorandom**.
Los números pseudorandom no son verdaderamente aleatorios porque se generan mediante un cálculo determinista, pero con solo mirar los números es prácticamente imposible distinguirlos de números aleatorios.

El módulo `random` proporciona funciones que generan números pseudorandom -- a los que de aquí en adelante simplemente llamaré "random".
Podemos importarlo así.

In [37]:
import random

In [38]:
# this cell initializes the random number generator so it
# generates the same sequence each time the notebook runs.

random.seed(4)

El módulo `random` proporciona una función llamada `choice` que elige un elemento de una lista al azar, con la misma probabilidad de elegir cada elemento.

In [48]:
t = [1, 2, 3]
random.choice(t)

1

Si llamas a la función otra vez, podrías obtener el mismo elemento de nuevo, o uno diferente.

In [43]:
random.choice(t)

2

A largo plazo, esperamos obtener cada elemento aproximadamente el mismo número de veces.

Si usas `choice` con un diccionario, obtienes un `KeyError`.

In [49]:
%%expect KeyError

random.choice(word_counter)

KeyError: 1644

Para elegir una clave aleatoria, tienes que poner las claves en una lista y luego llamar a `choice`.

In [51]:
words = list(word_counter)
random.choice(words)

'folly'

Si generamos una secuencia aleatoria de palabras, no tiene mucho sentido.

In [52]:
for i in range(6):
    word = random.choice(words)
    print(word, end=' ')

withdrew powers turpitude would hitherto familiar 

Parte del problema es que no estamos teniendo en cuenta que algunas palabras son más comunes que otras.
Los resultados serán mejores si elegimos palabras con distintos "weights", de modo que algunas se elijan más a menudo que otras.

Si usamos los valores de `word_counter` como weights, cada palabra se elige con una probabilidad que depende de su frecuencia.

In [53]:
weights = word_counter.values()

El módulo `random` proporciona otra función llamada `choices` que toma weights como argumento opcional.

In [54]:
random.choices(words, weights=weights)

['said']

Y toma otro argumento opcional, `k`, que especifica el número de palabras a seleccionar.

In [55]:
random_words = random.choices(words, weights=weights, k=6)
random_words

['to', 'was', 'and', 'a', 'lurking', 'musingly']

El resultado es una lista de strings que podemos unir en algo que se parece más a una oración.

In [56]:
' '.join(random_words)

'to was and a lurking musingly'

Si eliges palabras del libro al azar, obtienes una idea del vocabulario, pero una serie de palabras aleatorias rara vez tiene sentido porque no hay relación entre palabras sucesivas.
Por ejemplo, en una oración real esperas que un artículo como "the" vaya seguido de un adjetivo o un sustantivo, y probablemente no de un verbo o adverbio.
Así que el siguiente paso es observar estas relaciones entre palabras.

## Bigramas

En lugar de mirar una palabra cada vez, ahora miraremos secuencias de dos palabras, que se llaman **bigramas**.
Una secuencia de tres palabras se llama **trigram**, y una secuencia con un número no especificado de palabras se llama **n-gram**.

Escribamos un programa que encuentre todos los bigramas del libro y el número de veces que aparece cada uno.
Para guardar los resultados, usaremos un diccionario donde

* Las claves son tuplas de strings que representan bigramas, y

* Los valores son enteros que representan frecuencias.

Llamémoslo `bigram_counter`.

In [57]:
bigram_counter = {}

La siguiente función toma una lista de dos strings como parámetro.
Primero crea una tupla con los dos strings, que puede usarse como clave en un diccionario.
Luego añade la clave a `bigram_counter`, si no existe, o incrementa la frecuencia si existe.

In [58]:
def count_bigram(bigram):
    key = tuple(bigram)
    if key not in bigram_counter:
        bigram_counter[key] = 1
    else:
        bigram_counter[key] += 1

A medida que recorremos el libro, tenemos que llevar la cuenta de cada par de palabras consecutivas.
Así que si vemos la secuencia "man is not truly one", añadiríamos los bigramas "man is", "is not", "not truly", y así sucesivamente.

Para llevar la cuenta de estos bigramas, usaremos una lista llamada `window`, porque es como una ventana que se desliza sobre las páginas del libro, mostrando solo dos palabras a la vez.
Inicialmente, `window` está vacía.

In [59]:
window = []

Usaremos la siguiente función para procesar las palabras una por una.

In [60]:
def process_word(word):
    window.append(word)

    if len(window) == 2:
        count_bigram(window)
        window.pop(0)

La primera vez que se llama a esta función, añade la palabra dada a `window`.
Como solo hay una palabra en la window, todavía no tenemos un bigrama, así que la función termina.

La segunda vez que se llama -- y todas las veces después -- añade una segunda palabra a `window`.
Como hay dos palabras en la window, llama a `count_bigram` para llevar la cuenta de cuántas veces aparece cada bigrama.
Luego usa `pop` para eliminar la primera palabra de la window.

El siguiente programa hace un bucle por las palabras del libro y las procesa una a una.

In [61]:
for line in open(filename):
    for word in split_line(line):
        word = clean_word(word)
        process_word(word)

El resultado es un diccionario que asocia cada bigrama con el número de veces que aparece.
Podemos usar `print_most_common` para ver los bigramas más comunes.

In [62]:
print_most_common(bigram_counter)

178	('of', 'the')
139	('in', 'the')
94	('it', 'was')
80	('and', 'the')
73	('to', 'the')


Al mirar estos resultados, podemos hacernos una idea de qué pares de palabras tienen más probabilidad de aparecer juntos.
También podemos usar los resultados para generar texto aleatorio, así.

In [63]:
random.seed(0)

In [64]:
bigrams = list(bigram_counter)
weights = bigram_counter.values()
random_bigrams = random.choices(bigrams, weights=weights, k=6)

`bigrams` es una lista de los bigramas que aparecen en los libros.
`weights` es una lista de sus frecuencias, así que `random_bigrams` es una muestra donde la probabilidad de que se seleccione un bigrama es proporcional a su frecuencia.

Aquí están los resultados.

In [65]:
for pair in random_bigrams:
    print(' '.join(pair), end=' ')

begun to after this well hosts is if or above the laboratory 

Esta forma de generar texto es mejor que elegir palabras aleatorias, pero todavía no tiene mucho sentido.

## Análisis de Markov

Podemos hacerlo mejor con el análisis de texto mediante cadenas de Markov, que calcula, para cada palabra de un texto, la lista de palabras que vienen después.
Como ejemplo, analizaremos esta letra de la canción de Monty Python *Eric, the Half a Bee*:

In [66]:
song = """
Half a bee, philosophically,
Must, ipso facto, half not be.
But half the bee has got to be
Vis a vis, its entity. D'you see?
"""

Para guardar los resultados, usaremos un diccionario que asocia cada palabra con la lista de palabras que la siguen.

In [67]:
successor_map = {}

Como ejemplo, empecemos con las dos primeras palabras de la canción.

In [68]:
first = 'half'
second = 'a'

Si la primera palabra no está en `successor_map`, tenemos que añadir un nuevo elemento que asocie la primera palabra con una lista que contiene la segunda palabra.

In [69]:
successor_map[first] = [second]
successor_map

{'half': ['a']}

Si la primera palabra ya está en el diccionario, podemos buscarla para obtener la lista de sucesores que hemos visto hasta ahora, y añadir el nuevo.

In [70]:
first = 'half'
second = 'not'

successor_map[first].append(second)
successor_map

{'half': ['a', 'not']}

La siguiente función encapsula estos pasos.

In [71]:
def add_bigram(bigram):
    first, second = bigram

    if first not in successor_map:
        successor_map[first] = [second]
    else:
        successor_map[first].append(second)

Si el mismo bigrama aparece más de una vez, la segunda palabra se añade a la lista más de una vez.
De esta manera, `successor_map` lleva la cuenta de cuántas veces aparece cada sucesor.

Como hicimos en la sección anterior, usaremos una lista llamada `window` para guardar pares de palabras consecutivas.
Y usaremos la siguiente función para procesar las palabras una por una.

In [72]:
def process_word_bigram(word):
    window.append(word)

    if len(window) == 2:
        add_bigram(window)
        window.pop(0)

Así es como la usamos para procesar las palabras de la canción.

In [73]:
successor_map = {}
window = []

for word in song.split():
    word = clean_word(word)
    process_word_bigram(word)

Y aquí están los resultados.

In [74]:
successor_map

{'half': ['a', 'not', 'the'],
 'a': ['bee', 'vis'],
 'bee': ['philosophically', 'has'],
 'philosophically': ['must'],
 'must': ['ipso'],
 'ipso': ['facto'],
 'facto': ['half'],
 'not': ['be'],
 'be': ['but', 'vis'],
 'but': ['half'],
 'the': ['bee'],
 'has': ['got'],
 'got': ['to'],
 'to': ['be'],
 'vis': ['a', 'its'],
 'its': ['entity'],
 'entity': ["d'you"],
 "d'you": ['see']}

La palabra `'half'` puede ir seguida de `'a'`, `'not'` o `'the'`.
La palabra `'a'` puede ir seguida de `'bee'` o `'vis'`.
La mayoría de las demás palabras aparecen solo una vez, así que van seguidas de una única palabra.

Ahora analicemos el libro.

In [75]:
successor_map = {}
window = []

for line in open(filename):
    for word in split_line(line):
        word = clean_word(word)
        process_word_bigram(word)

Podemos buscar cualquier palabra y encontrar las palabras que pueden seguirla.

In [76]:
# I used this cell to find a predecessor with a good number of possible successors
# and at least one repeated word.

def has_duplicates(t):
    return len(set(t)) < len(t)

for key, value in successor_map.items():
    if len(value) == 7 and has_duplicates(value):
        print(key, value)

story ['of', 'of', 'indeed', 'but', 'for', 'of', 'that']
incident ['of', 'of', 'at', 'of', 'of', 'at', 'this']
lanyon’s ['narrative', 'there', 'face', 'manner', 'narrative', 'the', 'condemnation']
common ['it', 'interest', 'friends', 'friends', 'observers', 'but', 'quarry']
relief ['the', 'to', 'when', 'that', 'that', 'of', 'it']
appearance ['of', 'well', 'something', 'he', 'amply', 'of', 'which']
going ['east', 'in', 'to', 'to', 'up', 'to', 'of']
till ['at', 'the', 'i', 'yesterday', 'the', 'that', 'weariness']
walk ['and', 'into', 'with', 'all', 'steadfastly', 'attired', 'with']
sounds ['nothing', 'carried', 'out', 'the', 'of', 'of', 'with']
really ['like', 'damnable', 'can', 'a', 'a', 'not', 'be']
does ['not', 'not', 'indeed', 'not', 'the', 'not', 'not']
reply ['i', 'but', 'whose', 'i', 'some', 'that’s', 'i']
continued ['mr', 'the', 'the', 'the', 'the', 'poole', 'utterson']
seems ['scarcely', 'hardly', 'to', 'she', 'much', 'he', 'to']
walked ['on', 'over', 'some', 'was', 'on', 'with'

In [77]:
successor_map['going']

['east', 'in', 'to', 'to', 'up', 'to', 'of']

En esta lista de sucesores, observa que la palabra `'to'` aparece tres veces -- los demás sucesores aparecen solo una vez.

## Generar texto

Podemos usar los resultados de la sección anterior para generar texto nuevo con las mismas relaciones entre palabras consecutivas que en el original.
Así funciona:

* Empezando con cualquier palabra que aparezca en el texto, buscamos sus posibles sucesores y elegimos uno al azar.

* Luego, usando la palabra elegida, buscamos sus posibles sucesores y elegimos uno al azar.

Podemos repetir este proceso para generar tantas palabras como queramos.
Como ejemplo, empecemos con la palabra `'although'`.
Estas son las palabras que pueden seguirla.

In [78]:
word = 'although'
successors = successor_map[word]
successors

['i', 'a', 'it', 'the', 'we', 'they', 'i']

In [79]:
# this cell initializes the random number generator so it
# starts at the same point in the sequence each time this
# notebook runs.

random.seed(2)

Podemos usar `choice` para elegir de la lista con la misma probabilidad.

In [80]:
word = random.choice(successors)
word

'i'

Si la misma palabra aparece más de una vez en la lista, es más probable que sea seleccionada.

Repitiendo estos pasos, podemos usar el siguiente bucle para generar una serie más larga.

In [81]:
for i in range(10):
    successors = successor_map[word]
    word = random.choice(successors)
    print(word, end=' ')

continue to hesitate and swallowed the smile withered from that 

El resultado suena más como una oración real, pero todavía no tiene mucho sentido.

Podemos hacerlo mejor usando más de una palabra como clave en `successor_map`.
Por ejemplo, podemos crear un diccionario que asocie cada bigrama -- o trigram -- con la lista de palabras que vienen después.
Como ejercicio, tendrás la oportunidad de implementar este análisis y ver cómo son los resultados.

## Depuración

A estas alturas estamos escribiendo programas más sustanciales, y puede que descubras que pasas más tiempo depurando.
Si estás atascado con un bug difícil, aquí tienes algunas cosas que puedes probar:

* Leer: examina tu código, léelo en voz alta para ti y comprueba que dice lo que querías decir.

* Ejecutar: experimenta haciendo cambios y ejecutando versiones diferentes. A menudo, si muestras lo correcto en el lugar correcto del programa, el problema se vuelve obvio, pero a veces tienes que construir código de apoyo.

* Rumiar: ¡tómate tiempo para pensar! ¿Qué tipo de error es: de sintaxis, de runtime,
    o semántico? ¿Qué información puedes obtener de los mensajes de error,
    o del output del programa? ¿Qué tipo de error podría causar
    el problema que estás viendo? ¿Qué cambiaste por última vez, antes de que
    apareciera el problema?

* Rubberducking: si explicas el problema a otra persona, a veces encuentras la
    respuesta antes de terminar de hacer la pregunta. A menudo no necesitas
    a la otra persona; podrías hablarle a un patito de goma. Y ese es
    el origen de la estrategia conocida como **rubber duck
    depuración**. No me lo estoy inventando -- ver
    <https://en.wikipedia.org/wiki/Rubber_duck_depuración>.

* Retirarse: en algún momento, lo mejor es retroceder -- deshacer cambios
    recientes -- hasta llegar a un programa que funcione. Luego puedes empezar a reconstruir.
    
* Descansar: si le das un respiro a tu cerebro, a veces encontrará el problema por ti.

Los programadores principiantes a veces se quedan atascados en una de estas actividades y olvidan las demás. Cada actividad tiene su propio modo de fallo.

Por ejemplo, leer tu código funciona si el problema es un error tipográfico, pero no si el problema es un malentendido conceptual.
Si no entiendes lo que hace tu programa, puedes leerlo 100 veces y nunca ver el error, porque el error está en tu cabeza.

Ejecutar experimentos puede funcionar, especialmente si ejecutas tests pequeños y simples.
Pero si haces experimentos sin pensar ni leer tu código, puede llevar mucho tiempo averiguar qué está pasando.

Tienes que tomarte tiempo para pensar. Depurar es como una ciencia experimental. Deberías tener al menos una hipótesis sobre cuál es el problema. Si hay dos o más posibilidades, intenta pensar en un test que elimine una de ellas.

Pero incluso las mejores técnicas de depuración fallarán si hay demasiados errores, o si el código que intentas arreglar es demasiado grande y complicado.
A veces la mejor opción es retirarse, simplificando el programa hasta volver a algo que funcione.

Los programadores principiantes a menudo se resisten a retirarse porque no soportan borrar una línea de código (aunque esté mal). Si te hace sentir mejor, copia tu programa en otro archivo antes de empezar a recortarlo. Luego puedes copiar las piezas de vuelta una por una.

Encontrar un bug difícil requiere leer, ejecutar, rumiar, retirarse y a veces descansar.
Si te atascas en una de estas actividades, prueba las otras.

## Glosario

**valor por defecto:**
El valor asignado a un parámetro si no se proporciona ningún argumento.

**override:**
 Reemplazar un valor por defecto por un argumento.

**determinista:**
 Un programa determinista hace lo mismo cada vez que se ejecuta, dados los mismos inputs.

**pseudorandom:**
 Una secuencia pseudorandom de números parece aleatoria, pero es generada por un programa determinista.

**bigrama:**
Una secuencia de dos elementos, a menudo palabras.

**trigram:**
Una secuencia de tres elementos.

**n-gram:**
Una secuencia de un número no especificado de elementos.

**rubber duck depuración:**
Una forma de depurar explicando un problema en voz alta a un objeto inanimado.

## Ejercicios

In [82]:
# This cell tells Jupyter to provide detailed debugging information
# when a runtime error occurs. Run it before working on the exercises.

%xmode Verbose

Exception reporting mode: Verbose


### Pregunta a un asistente virtual

En `add_bigram`, la sentencia `if` crea una nueva lista o añade un elemento a una lista existente, dependiendo de si la clave ya está en el diccionario.

In [84]:
def add_bigram(bigram):
    first, second = bigram

    if first not in successor_map:
        successor_map[first] = [second]
    else:
        successor_map[first].append(second)

Diccionarios proporcionan un método llamado `setdefault` que podemos usar para hacer lo mismo de forma más concisa.
Pregunta a un asistente virtual cómo funciona, o copia `add_bigram` en un asistente virtual y pregunta "Can you rewrite this using `setdefault`?"

En este capítulo implementamos análisis y generación de texto con cadenas de Markov.
Si tienes curiosidad, puedes pedirle a un asistente virtual más información sobre el tema.
Una de las cosas que podrías aprender es que los asistentes virtuales usan algoritmos que son similares en muchos aspectos -- pero también diferentes en aspectos importantes.
Pregunta a un VA: "What are the differences between large language models like GPT and Markov chain text analysis?

In [85]:
# Can you rewrite this using setdefault?
def add_bigram(bigram):
    first, second = bigram
    successor_map.setdefault(first, []).append(second)

### What are the differences between large language models like GPT and Markov chain text analysis?

Una cadena de Markov es como mirar solo unas pocas palabras anteriores para decidir la siguiente.

Un LLM como GPT es como haber leído millones de libros y usar todo ese conocimiento para predecir la siguiente palabra teniendo en cuenta el contexto completo, la gramática, el significado y la intención del texto.

### Ejercicio

Escribe una función que cuente el número de veces que aparece cada trigram (secuencia de tres palabras).
Si pruebas tu función con el texto de _Dr. Jekyll and Mr. Hyde_, deberías encontrar que el trigram más común es "said the lawyer".

Pista: escribe una función llamada `count_trigram` que sea similar a `count_bigram`. Luego escribe una función llamada `process_word_trigram` que sea similar a `process_word_bigram`.

In [86]:
# Solution goes here
trigram_counter = {}
def count_trigram(trigram):
    key = tuple(trigram)
    if key not in trigram_counter:
        trigram_counter[key] = 1
    else:
        trigram_counter[key] += 1

In [90]:
# version mas corta
def count_trigram(trigram):
    key = tuple(trigram)
    trigram_counter[key] = trigram_counter.get(key, 0) + 1

In [91]:
# Solution goes here
def process_word_trigram(word):
    window.append(word)

    if len(window) == 3:
        count_trigram(window)
        window.pop(0)

Puedes usar el siguiente bucle para leer el libro y procesar las palabras.

In [92]:
trigram_counter = {}
window = []

for line in open(filename):
    for word in split_line(line):
        word = clean_word(word)
        process_word_trigram(word)

Luego usa `print_most_common` para encontrar los trigrams más comunes del libro.

In [93]:
print_most_common(trigram_counter)

20	('said', 'the', 'lawyer')
16	('said', 'mr', 'utterson')
15	('of', 'edward', 'hyde')
13	('the', 'name', 'of')
11	('it', 'was', 'a')


### Ejercicio

Ahora implementemos análisis de texto con cadenas de Markov usando un mapeo de cada bigrama a una lista de posibles sucesores.

Empezando con `add_bigram`, escribe una función llamada `add_trigram` que tome una lista de tres palabras y añada o actualice un elemento en `successor_map`, usando las dos primeras palabras como clave y la tercera palabra como posible sucesor.

In [99]:
# Solution goes here
def add_trigram(trigram):
    first, second, third = trigram

    if (first, second) not in successor_map:
        successor_map[first, second] = [third]
    else:
        successor_map[first, second].append(third)

Aquí tienes una versión de `process_word_trigram` que llama a `add_trigram`.

In [100]:
def process_word_trigram(word):
    window.append(word)

    if len(window) == 3:
        add_trigram(window)
        window.pop(0)

Puedes usar el siguiente bucle para probar tu función con la letra de "Eric, the Half a Bee".

In [101]:
successor_map = {}
window = []

for string in song.split():
    word = string.strip(punctuation).lower()
    process_word_trigram(word)

Si tu función funciona como se espera, el predecesor `('half', 'a')` debería asociarse con una lista cuyo único elemento es `'bee'`.
De hecho, resulta que cada bigrama de esta canción aparece solo una vez, así que todos los valores de `successor_map` tienen un único elemento.

In [102]:
successor_map

{('half', 'a'): ['bee'],
 ('a', 'bee'): ['philosophically'],
 ('bee', 'philosophically'): ['must'],
 ('philosophically', 'must'): ['ipso'],
 ('must', 'ipso'): ['facto'],
 ('ipso', 'facto'): ['half'],
 ('facto', 'half'): ['not'],
 ('half', 'not'): ['be'],
 ('not', 'be'): ['but'],
 ('be', 'but'): ['half'],
 ('but', 'half'): ['the'],
 ('half', 'the'): ['bee'],
 ('the', 'bee'): ['has'],
 ('bee', 'has'): ['got'],
 ('has', 'got'): ['to'],
 ('got', 'to'): ['be'],
 ('to', 'be'): ['vis'],
 ('be', 'vis'): ['a'],
 ('vis', 'a'): ['vis'],
 ('a', 'vis'): ['its'],
 ('vis', 'its'): ['entity'],
 ('its', 'entity'): ["d'you"],
 ('entity', "d'you"): ['see']}

Puedes usar el siguiente bucle para probar tu función con las palabras del libro.

In [104]:
successor_map = {}
window = []

for line in open(filename):
    for word in split_line(line):
        word = clean_word(word)

        if word == '':
            continue

        process_word_trigram(word)

En el siguiente ejercicio, usarás los resultados para generar nuevo texto aleatorio.

### Ejercicio

Para este ejercicio, asumiremos que `successor_map` es un diccionario que asocia cada bigrama con la lista de palabras que lo siguen.

In [105]:
# this cell initializes the random number generator so it
# starts at the same point in the sequence each time this
# notebook runs.

random.seed(3)

Para generar texto aleatorio, empezaremos eligiendo una clave aleatoria de `successor_map`.

In [106]:
successors = list(successor_map)
bigram = random.choice(successors)
bigram

('doubted', 'if')

Ahora escribe un bucle que genere 50 palabras más siguiendo estos pasos:

1. En `successor_map`, busca la lista de palabras que pueden seguir a `bigram`.

2. Elige una de ellas al azar e imprímela.

3. Para la siguiente iteración, crea un nuevo bigrama que contenga la segunda palabra de `bigram` y el sucesor elegido.

Por ejemplo, si empezamos con el bigrama `('doubted', 'if')` y elegimos `'from'` como sucesor, el siguiente bigrama es `('if', 'from')`.

In [107]:
# Solution goes here
for i in range(50):
    successors = successor_map[bigram]

    next_word = random.choice(successors)

    print(next_word, end=' ')

    bigram = (bigram[1], next_word)

from that day forth utterson desired the society of his ape-like spite and indeed when the night was fully come he set it down to dinner why do you think before you answer for it god bless me the deadliest terror sits by me at the table and still he 

Si todo funciona, deberías encontrar que el texto generado es reconociblemente similar en estilo al original, y algunas frases tienen sentido, pero el texto podría saltar de un tema a otro.

Como ejercicio extra, modifica tu solución a los dos últimos ejercicios para usar trigrams como claves en `successor_map`, y mira qué efecto tiene en los resultados.

In [108]:
print(bigram[0], bigram[1], end=' ')

for i in range(50):
    successors = successor_map[bigram]
    next_word = random.choice(successors)

    print(next_word, end=' ')

    bigram = (bigram[1], next_word)

still he was perhaps relieved to be born and at the horror of being hyde that racked me i have been about half full of premature twilight although the sky high up overhead was still rolling in through the by-street and that a mr hyde had numbered few familiars even the nightmares 

[Think Python: 3rd Edition](https://allendowney.github.io/ThinkPython/index.html)

Copyright 2024 [Allen B. Downey](https://allendowney.com)

Traducción al español por midudev (Miguel Ángel Durán).

Código license: [MIT License](https://mit-license.org/)

Text license: [Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International](https://creativecommons.org/licenses/by-nc-sa/4.0/)